# core

> Core module for lazyq — a lightweight, chainable query pipeline for Python.

In [19]:
#| default_exp core

In [20]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Sentinels
SKIP = object()
STOP = object()

# Stream Operations
class Map:
    def __init__(self, fn): self.fn = fn
    def __call__(self, x): return self.fn(x)

class Filter:
    def __init__(self, fn):
        if not callable(fn): raise TypeError('Filter expects a callable')
        self.fn = fn
    def __call__(self, item):
        return item if self.fn(item) else SKIP

class Limit:
    def __init__(self, n): self.n = n; self.count = 0
    def __call__(self, item):
        if self.count >= self.n: return STOP
        self.count += 1
        return item

class Select:
    def __init__(self, keys): self.keys = keys if isinstance(keys, list) else [keys]
    def __call__(self, item): return {k: item[k] for k in self.keys}

class GroupBy:
    def __init__(self, key): self.key = key
    def run(self, data):
        group = {}
        for item in data:
            k = item[self.key]
            if k not in group: group[k] = []
            group[k].append(item)
        return group.items()

class Reduce:
    def __init__(self, fn, initial=None):
        self.fn = fn
        self.initial = initial
    def run(self, data):
        it = iter(data)
        if self.initial is None:
            try: acc = next(it)
            except StopIteration: return []
        else: acc = self.initial
        for item in it:
            acc = self.fn(acc, item)
        return [acc]

class Sort:
    def __init__(self, key=None, reverse=False):
        self.key = key
        self.reverse = reverse
    def run(self, data):
        if self.key is None:
            return sorted(data, reverse=self.reverse)
        if callable(self.key):
            return sorted(data, key=self.key, reverse=self.reverse)
        return sorted(data, key=lambda x: x[self.key], reverse=self.reverse)

# Filter Expression Classes
class Condition:
    def __init__(self, fn): self.fn = fn
    def __call__(self, item): return self.fn(item)
    def __and__(self, other): return Condition(lambda x: self(x) and other(x))
    def __or__(self, other): return Condition(lambda x: self(x) or other(x))

class F:
    def __init__(self, name): self.name = name
    def __gt__(self, value): return Condition(lambda x: x[self.name] > value)
    def __lt__(self, value): return Condition(lambda x: x[self.name] < value)
    def __eq__(self, value): return Condition(lambda x: x[self.name] == value)
    def __ne__(self, value): return Condition(lambda x: x[self.name] != value)
    def __le__(self, value): return Condition(lambda x: x[self.name] <= value)
    def __ge__(self, value): return Condition(lambda x: x[self.name] >= value)
    def not_null(self): return Condition(lambda x: x.get(self.name) is not None)
    def is_null(self): return Condition(lambda x: x.get(self.name) is None)

# File readers
import csv, sqlite3, json

def read_csv(path):
    with open(path) as f:
        for row in csv.DictReader(f): yield row

def read_sqlite3(db_path, query):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        cols = [col[0] for col in cursor.description]
        for row in cursor: yield dict(zip(cols, row))

def read_json(path, key=None, stream=False):
    if not stream:
        with open(path) as f:
            data = json.load(f)
            if key: data = data.get(key, [])
            if isinstance(data, dict): yield data
            else:
                for item in data: yield item
    else:
        import ijson
        with open(path, 'rb') as f:
            prefix = f"{key}.item" if key else "item"
            for item in ijson.items(f, prefix): yield item

def read_yaml(path, key=None):
    import yaml
    with open(path) as f:
        data = yaml.safe_load(f)
        if key:
            if not isinstance(data, dict): raise ValueError("Key can only be used with dict-like data")
            data = data.get(key, [])
        if isinstance(data, dict): yield data
        else:
            for item in data: yield item

def read_excel(path, sheet_name=None):
    from openpyxl import load_workbook
    wb = load_workbook(path, read_only=True)
    ws = wb[sheet_name] if sheet_name else wb.active
    rows = ws.iter_rows(values_only=True)
    headers = next(rows)
    for row in rows: yield dict(zip(headers, row))

# Query classes
class Query:
    def __init__(self, data, operations=None):
        self.data = data
        self.operations = operations or []

    def map(self, fn): return Query(self.data, self.operations + [(lambda: Map(fn), 'map')])
    def filter(self, fn): return Query(self.data, self.operations + [(lambda: Filter(fn), 'filter')])
    def limit(self, n): return Query(self.data, self.operations + [(lambda: Limit(n), 'limit')])
    def select(self, keys): return Query(self.data, self.operations + [(lambda: Select(keys), f'select({keys})')])
    def groupby(self, key): return GroupedQuery(self.data, self.operations + [(lambda: GroupBy(key), f'groupby({key})')])
    def reduce(self, fn, initial=None): return Query(self.data, self.operations + [(lambda: Reduce(fn, initial), 'reduce')])
    def count(self): return self.reduce(lambda acc, _: acc + 1, initial=0)
    def sum(self, by=None):
        name = "sum" if by is None else f"sum({by})"
        if by is None:
            return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: acc + x, 0), name)])
        return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: acc + x[by], 0), name)])
    def max(self, by=None):
        name = "max" if by is None else f"max({by})"
        if by is None:
            return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: x if x > acc else acc), name)])
        return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: x if x[by] > acc[by] else acc), name)])
    def min(self, by=None):
        name = "min" if by is None else f"min({by})"
        if by is None:
            return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: x if x < acc else acc), name)])
        return Query(self.data, self.operations + [(lambda: Reduce(lambda acc, x: x if x[by] < acc[by] else acc), name)])
    def sort(self, key=None, reverse=False):
        name = ("sort(desc)" if reverse else "sort") if key is None else (f"sort({key}, desc)" if reverse else f"sort({key})")
        return Query(self.data, self.operations + [(lambda: Sort(key, reverse), name)])

    def collect(self, n=None): return list(self) if n is None else list(self.limit(n))

    def __repr__(self):
        if not self.operations: return "Query()"
        return f"Query(steps={len(self.operations)}: " + " -> ".join(name for _, name in self.operations) + ")"

    def show(self, n=5):
        rows = list(self)[:n]
        if not rows: print("(empty)"); return
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
            widths = {h: max(len(str(h)), *(len(str(r.get(h, ""))) for r in rows)) for h in headers}
            header_line = " | ".join(f"{h:<{widths[h]}}" for h in headers)
            print(header_line)
            print("-" * len(header_line))
            for r in rows:
                print(" | ".join(f"{str(r.get(h, '')):<{widths[h]}}" for h in headers))
        elif isinstance(rows[0], tuple):
            for r in rows: print(r)
        else:
            for r in rows: print(r)

    def __iter__(self):
        data = self.data() if callable(self.data) else self.data
        ops = [f() for f, _ in self.operations]
        for op in ops:
            if hasattr(op, 'run'): data = op.run(data)
            else: data = self._apply_stram_op(data, op)
        for item in data: yield item

    def _apply_stram_op(self, data, op):
        for item in data:
            item = op(item)
            if item is SKIP: continue
            if item is STOP: return
            yield item

    @classmethod
    def from_iterable(cls, data): return cls(data)
    @classmethod
    def from_csv(cls, path): return cls(lambda: read_csv(path))
    @classmethod
    def from_sqlite(cls, db_path, table): return cls(lambda: read_sqlite3(db_path, f"SELECT * FROM {table}"))
    @classmethod
    def from_sqlite_query(cls, db_path, query): return cls(lambda: read_sqlite3(db_path, query))
    @classmethod
    def from_json(cls, path, key=None, stream=False): return cls(lambda: read_json(path, key, stream))
    @classmethod
    def from_yaml(cls, path, key=None): return cls(lambda: read_yaml(path, key))
    @classmethod
    def from_excel(cls, path, sheet_name=None): return cls(lambda: read_excel(path, sheet_name))
    @classmethod
    def tools(cls):

        sources = {
            "from_iterable(data)": "Create Query from list/iterable",
            "from_csv(path)": "Load data from CSV file",
            "from_sqlite(db, table)": "Load table from SQLite database",
            "from_sqlite_query(db, query)": "Run custom SQL query on SQLite database",
            "from_json(path, key=None, stream=False)": "Load data from JSON file",
            "from_yaml(path, key=None)": "Load data from YAML file",
            "from_excel(path, sheet_name=None)": "Load data from Excel file",
        }

        transforms = {
            "map(fn)": "Transform each row",
            "filter(fn)": "Filter rows by condition",
            "limit(n)": "Limit number of rows",
            "sort(key, reverse=False)": "Sort rows",
            "groupby(key)": "Group rows by a key",
        }

        aggregations = {
            "sum(by=None)": "Sum values",
            "count()": "Count rows",
            "max(by=None)": "Maximum value/row",
            "min(by=None)": "Minimum value/row",
        }

        output = {
            "collect(n=None)": "Return results as list",
            "show(n=5)": "Pretty print results",
        }

        print("\n📦 Data Sources:\n")
        for name, desc in sources.items():
            print(f"{name:<35} → {desc}")

        print("\n🔄 Transformations:\n")
        for name, desc in transforms.items():
            print(f"{name:<35} → {desc}")

        print("\n📊 Aggregations:\n")
        for name, desc in aggregations.items():
            print(f"{name:<35} → {desc}")

        print("\n📤 Output:\n")
        for name, desc in output.items():
            print(f"{name:<35} → {desc}")


class GroupedQuery(Query):
    def __init__(self, data, operations): super().__init__(data, operations)
    def map(self, fn): return GroupedQuery(self.data, self.operations + [(lambda: Map(fn), 'map')])
    def count(self): return self.map(lambda g: (g[0], len(g[1])))
    def sum(self, by): return self.map(lambda g: (g[0], sum(float(i[by]) for i in g[1])))
    def take(self, n): return self.map(lambda g: (g[0], g[1][:n]))
    def max(self, by): return self.map(lambda g: (g[0], max(g[1], key=lambda i: i[by])))
    def min(self, by): return self.map(lambda g: (g[0], min(g[1], key=lambda i: i[by])))

In [22]:
#| hide
import nbdev; nbdev.nbdev_export()